# Concurrency – Practice Exercises

This notebook contains hands-on exercises to practice **concurrent.futures** (threads and processes),
following the concepts from `g_concurrency.ipynb`.

For each exercise:

- Read the description carefully.
- Implement your solution in the provided code cell.
- Use timing (e.g. `time.perf_counter()`) where relevant and add a short demo or print to verify.

### Contents

- [Exercise 1 – Submit and Result](#exercise-1--submit-and-result)
- [Exercise 2 – Sequential vs ThreadPool Timing](#exercise-2--sequential-vs-threadpool-timing)
- [Exercise 3 – executor.map](#exercise-3--executormap)
- [Exercise 4 – ProcessPoolExecutor for CPU Work](#exercise-4--processpoolexecutor-for-cpu-work)
- [Exercise 5 – as_completed](#exercise-5--as_completed)
- [Exercise 6 – Exception Handling from Futures](#exercise-6--exception-handling-from-futures)
- [Exercise 7 – wait with FIRST_EXCEPTION](#exercise-7--wait-with-first_exception)
- [Exercise 8 – result(timeout)](#exercise-8--resulttimeout)
- [Exercise 9 – cancel()](#exercise-9--cancel)
- [Exercise 10 – I/O and Timeout Together](#exercise-10--io-and-timeout-together)

## Exercise 1 – Submit and Result

**Goal**: Use **ThreadPoolExecutor** to run a simple function that takes a name and a delay, sleeps, then returns a string. Submit 2–3 tasks and print each result with `.result()`.

Requirements:

- Use `with ThreadPoolExecutor(max_workers=...) as executor`.
- Call `executor.submit(func, arg1, arg2)` for each task and store the returned futures.
- For each future, call `.result()` and print the return value.

Hint: The executor’s context manager waits for submitted work to finish on exit; call `.result()` before that to collect results.

In [ ]:
# Exercise 1 – submit and result

import time, requests
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# Simple GET to a public crypto endpoint

class Binance:
    URL_BASE = "https://api.binance.com/api/v3"
    @classmethod
    def get_price(cls, symbol: str) -> float:
        url = cls.URL_BASE + "/ticker/price"
        resp = requests.get(url, {"symbol": symbol})
        return float(resp.json()["price"])

symbols = ["BTC", "ETH", "DOGE", "LTC",
          "XRP", "AVAX", "USDC", "BNB"]
results_mth = dict.fromkeys(symbols)
results_mpr = dict.fromkeys(symbols)
results_seq = dict.fromkeys(symbols)

t = time.time()
with ThreadPoolExecutor(max_workers = len(symbols)) as pool:
    for s in symbols: results_mth[s] = pool.submit(Binance.get_price, s + "USDT")
delay_mth = time.time() - t

t = time.time()
with ThreadPoolExecutor(max_workers = len(symbols)) as pool:
    for s in symbols: results_mpr[s] = pool.submit(Binance.get_price, s + "USDT")
delay_mpr = time.time() - t

t = time.time()
for s in symbols: results_seq[s] = Binance.get_price(s + "USDT")
delay_seq = time.time() - t

results_mth = {symbol: r.result() for symbol, r in results_mth.items()}
print(f"Multi-thread tasks in: {delay_mth:.2f} secs...\n{results_mth}")
results_mpr = {symbol: r.result() for symbol, r in results_mpr.items()}
print(f"Multi-process tasks in: {delay_mpr:.2f} secs...\n{results_mpr}")
print(f"Sequential tasks in: {delay_seq:.2f} secs...\n{results_seq}")

# Demo: 2–3 tasks, print their results

## Exercise 2 – Sequential vs ThreadPool Timing

**Goal**: Run N simulated I/O tasks (e.g. each does `time.sleep(0.2)`) **sequentially** and measure total time; then run the **same N tasks** with **ThreadPoolExecutor** and measure again. Print both times and the speedup (sequential_time / thread_time).

Requirements:

- Define a small function that sleeps for a fixed delay and returns something (e.g. task id).
- Sequential: loop and call the function N times; time with `time.perf_counter()`.
- ThreadPool: submit N tasks, collect results with `.result()`, time the whole block.

Hint: With N workers and N tasks of 0.2 s each, thread total time should be close to 0.2 s.

In [ ]:
# Exercise 2 – sequential vs ThreadPool timing

N = 5
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)
results_mth = list(range(N))
results_seq = list(range(N))
futures = list(range(N))

t0 = int(time.time() * 1e6)
with ThreadPoolExecutor(max_workers = N) as pool:
    for n in range(N): futures[n] = pool.submit(sleep, 1)

t = time.perf_counter()
for n in range(N): results_mth[n] = futures[n].result() - t0
delay_mth = time.perf_counter() - t

t = time.perf_counter()
t0 = int(time.time() * 1e6)
for n in range(N): results_seq[n] = sleep(1) - t0
delay_seq = time.perf_counter() - t

print(f"MultiThread tasks in: {delay_mth:.2f} secs...\n{results_mth} ms")
print(f"Sequential tasks in: {delay_seq:.2f} secs...\n{results_seq} ms")

# Demo: e.g. 5 tasks × 0.2 s → sequential ~1 s, thread pool ~0.2 s

## Exercise 3 – executor.map

**Goal**: Run the same I/O-bound workload (e.g. N tasks with a short sleep) using **executor.map** instead of submit. Show that results come back **in the same order** as the input iterable.

Requirements:

- Use `executor.map(func, iterable)` so each element of the iterable is passed to `func`.
- Collect results with `list(executor.map(...))` and print them (or their order).

Hint: `map` yields results in submission order; use it when order matters.

In [ ]:
# Exercise 3 – executor.map

N = 5
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)
iterator = list(range(N))
results_mth = list(range(N))
results_seq = list(range(N))

t0 = int(time.time() * 1e6)
with ThreadPoolExecutor(max_workers = N) as pool:
    iterator = pool.map(sleep, [1] * N)

t = time.perf_counter()
for n, r in enumerate(iterator): results_mth[n] = r - t0
delay_mth = time.perf_counter() - t

t = time.perf_counter()
t0 = int(time.time() * 1e6)
for n in range(N): results_seq[n] = sleep(1) - t0
delay_seq = time.perf_counter() - t

print(f"MultiThread tasks in: {delay_mth:.2f} secs...\n{results_mth} ms")
print(f"Sequential tasks in: {delay_seq:.2f} secs...\n{results_seq} ms")

# Demo: e.g. 5 tasks × 0.2 s → sequential ~1 s, thread pool ~0.2 s

# Demo: map over range(5), show result order matches [0,1,2,3,4]

## Exercise 4 – ProcessPoolExecutor for CPU Work

**Goal**: Define a **CPU-heavy** function (e.g. a loop that does a lot of arithmetic) and run it for several inputs **sequentially** and then with **ProcessPoolExecutor**. Compare total times and print a simple speedup.

Requirements:

- The worker function must be **defined at top level** (or in the same cell) so it can be pickled for the process pool.
- Use `with ProcessPoolExecutor(max_workers=...) as executor` and submit one task per input.

Hint: In Jupyter, define the CPU function in the same cell (or a cell above) so it’s picklable; avoid lambdas for process pools.

In [3]:
# Exercise 4 – ProcessPoolExecutor for CPU

import os, sys, time
sys.path.insert(0, os.getcwd())
from concurrent.futures import ProcessPoolExecutor
from p_concurrency_ex4 import ols_test

args = [(int(1e8 / dim), dim) for dim in range(1, 4)]

# Sequential

t0 = time.perf_counter()

results_seq = [ols_test(*args) for args in args]
delay_seq = time.perf_counter() - t0

# Process pool

t0 = time.perf_counter()
results_mpr = list()
with ProcessPoolExecutor(max_workers = 4) as pool:
    futures = [pool.submit(ols_test, *args) for args in args]
    for f in futures: results_mpr.append(f.result())
delay_mpr = time.perf_counter() - t0

print(f"Delay for sequential process: {1e6 * delay_seq:.0f} us...\nResults: {results_seq}")
print(f"Delay for multi-core process: {1e6 * delay_mpr:.0f} us...\nResults: {results_mpr}")

Delay for sequential process: 7297628 us...
Results: [[-47.11544406754275], [-9.400905677959242, 22.621963511040878], [-21.927919258555995, -22.446563393281572, 13.995909811614867]]
Delay for multi-core process: 3528385 us...
Results: [[-47.11544406754275], [-9.400905677959242, 22.621963511040878], [-21.927919258555995, -22.446563393281572, 13.995909811614867]]


## NOTE: **Why put the worker in a separate module?**

When using `ProcessPoolExecutor` on Windows, Python starts worker processes with the `spawn` method. That means each worker must be able to import your worker function in a fresh interpreter, and it can’t reliably rely on definitions that only exist inside a Jupyter notebook’s `__main__` state. If the worker is not importable/picklable, the child process can fail during startup or task deserialization, which often shows up as `BrokenProcessPool`. 

Moving the worker into a real `.py` file (like `p_concurrency_ex4.py`) guarantees it is a top-level symbol that workers can import by module name. This makes failures deterministic and prevents the pool from crashing mid-run. It also keeps the notebook cleaner and the worker reusable in other contexts.

----

## Exercise 5 – as_completed

**Goal**: Submit several I/O tasks with **different delays** (e.g. 0.3, 0.1, 0.2 s). Use **as_completed(futures)** to iterate over futures as they finish and print each result (or task id) as it completes. The print order should reflect **completion order**, not submission order.

Requirements:

- Build a list (or dict) of futures from executor.submit.
- Loop over `as_completed(futures)` and call `.result()` on each future; print the result.

Hint: You can use a dict mapping future -> task_id so you know which task completed when you get the future from as_completed.

In [5]:
# Exercise 5 – as_completed

from concurrent.futures import ThreadPoolExecutor, as_completed

delays = [3, 1, 2, 0.5]
def sleep(n: float):
    time.sleep(n := abs(n))
    return int(time.time() * 1e6)

futures = list()
results_mth = list()
results_seq = list()
t0 = int(time.time() * 1e6)
with ThreadPoolExecutor(4) as pool:
    for n, delay in enumerate(delays):
        futures.append(pool.submit(sleep, delay))
    for future in as_completed(futures):
        results_mth.append(future.result() - t0)
delay_mth = int(time.time() * 1e6) - t0

t0 = int(time.time() * 1e6)
for n, delay in enumerate(delays):
    results_seq.append(sleep(delay) - t0)
delay_seq = int(time.time() * 1e6) - t0

print(f"Delay for sequential process: {delay_seq:.0f} us...\nResults: {results_seq}")
print(f"Delay for multithread process: {delay_mth:.0f} us...\nResults: {results_mth}")
print("Both multithread and sequential results should be sorted.")

# Demo: 3–5 tasks with different delays; print completion order

Delay for sequential process: 6510232 us...
Results: [3000558, 4001496, 6003007, 6507835]
Delay for multithread process: 3002531 us...
Results: [501561, 1001025, 2001973, 3000634]
Both multithread and sequential results should be sorted.


## Exercise 6 – Exception Handling from Futures

**Goal**: Submit a few tasks where **one** (or more) can **raise an exception** (e.g. raise if task_id == 2). When collecting results, use **future.exception()** to check for errors: if it’s not None, print the exception; otherwise print the result.

Requirements:

- Worker function: e.g. sleep a bit, then raise ValueError for a specific input.
- For each future, call `future.exception()`; if None, use `future.result()`, else print the exception.

Hint: Calling `.exception()` returns the exception object instead of re-raising it; use that to avoid crashing and to log failures.

In [19]:
# Exercise 6 – exception handling

from concurrent.futures import ThreadPoolExecutor, Future

delays = [3, 1, -2, 0.5]
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)

error = "Error for {} #{}/{}: \"{}\"".format

futures = list()
results_mth = list()
results_seq = list()
t0 = int(time.time() * 1e6)
with ThreadPoolExecutor(4) as pool:
    for n, delay in enumerate(delays):
        futures.append(pool.submit(sleep, delay))

for n, delay in enumerate(delays):
    future: Future = futures[n]
    if future.exception() is not None:
        error_str = repr(future.exception())
        print(error("MTH", n, delay, error_str))
    else:
        results_mth.append(future.result() - t0)
delay_mth = int(time.time() * 1e6) - t0

t0 = int(time.time() * 1e6)
for n, delay in enumerate(delays):
    try: results_seq.append(sleep(delay) - t0)
    except Exception as EXC:
        error_str = repr(EXC)
        print(error("SEQ", n, delay, error_str))
delay_seq = int(time.time() * 1e6) - t0

print(f"Delay for sequential process: {delay_seq:.0f} us...\nResults: {results_seq}")
print(f"Delay for multithread process: {delay_mth:.0f} us...\nResults: {results_mth}")

# Demo: 4 tasks, task 2 raises; print ok/error for each

Error for MTH #2/-2: "ValueError('sleep length must be non-negative')"
Error for SEQ #2/-2: "ValueError('sleep length must be non-negative')"
Delay for sequential process: 4501583 us...
Results: [3000228, 4000486, 4501486]
Delay for multithread process: 3001716 us...
Results: [3001065, 1001168, 501239]


## Exercise 7 – wait with FIRST_EXCEPTION

**Goal**: Use **wait(futures, return_when=FIRST_EXCEPTION)** so that `wait` returns as soon as **any** future raises. Then iterate over the `done` set: for each future, check `.exception()` and either print the result or the exception. Print how many futures are still `not_done`.

Requirements:

- Submit several tasks, one of which raises (e.g. after a short delay).
- Call `done, not_done = wait(futures, return_when=FIRST_EXCEPTION)`.
- Process `done` and report `len(not_done)`.

Hint: Import `wait` and `FIRST_EXCEPTION` from `concurrent.futures`.

In [ ]:
# Exercise 7 – wait with FIRST_EXCEPTION

from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import wait, FIRST_EXCEPTION

delays = {"tiny": 1e-4, "big": 3, "small": 1, "error": -1}
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)

t0 = int(time.time() * 1e6)
futures = dict()
with ThreadPoolExecutor(4) as pool:
    for key, delay in delays.items():
        futures[pool.submit(sleep, delay)] = key
    done, fail = wait(futures, return_when = FIRST_EXCEPTION)
    keys_done = [futures[future] for future in done]
    keys_fail = [futures[future] for future in fail]
delay = int(time.time() * 1e6) - t0

print(f"Done: {keys_done}... Failed: {keys_fail}... Delay: {delay:.0f} us")
print("""Note: if "tiny" is too small (< 1e-4), it can appear in "done".
      (Sometimes may finish its run before the error triggers)""")

# Demo: 5 tasks, one raises; show done set and not_done count

Done: ['tiny', 'error']... Failed: ['small', 'big']... Delay: 3001185 us
Note: if "tiny" is too small (< 1e-4), it can appear in "done".
      (Sometimes may finish its run before the error triggers)


## Exercise 8 – result(timeout)

**Goal**: Submit a **long-running** task (e.g. sleep 2 seconds). Call **future.result(timeout=0.5)** and catch **TimeoutError**; print a message that the task did not finish in time. Optionally submit a second, quick task and show that its result is available within the timeout.

Requirements:

- Use a worker that sleeps longer than the timeout you pass to `.result(timeout=...)`.
- Use try/except TimeoutError.

Hint: `.result(timeout=T)` blocks at most T seconds; after that it raises TimeoutError and the future may still be running.

In [ ]:
# Exercise 8 – result(timeout)

from concurrent.futures import ThreadPoolExecutor, wait

delays = {"tiny": 1e-4, "big": 3, "small": 1, "error": -1}
timeout = max(delays.values())
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)

future: Future = None
futures = dict()
t0 = int(time.time() * 1e6)
keys_done, keys_fail = list(), list()
with ThreadPoolExecutor(4) as pool:
    for key, delay in delays.items():
        futures[pool.submit(sleep, delay)] = key
    done, fail = wait(futures, timeout = timeout)
    for future, key in futures.items():
        if future.done():
            if not future.exception():
                keys_done.append(key)
            else: keys_fail.append(key)
        else: keys_fail.append(key)
        
delay = int(time.time() * 1e6) - t0

print(f"Done: {keys_done}... Failed: {keys_fail}... Delay: {delay:.0f} us")

# Demo: slow task 2 s, result(timeout=0.5) -> TimeoutError; fast task -> ok

Done: ['tiny']... Failed: ['big', 'small', 'error']... Delay: 3001151 us


## Exercise 9 – cancel()

**Goal**: Submit a **slow** task (e.g. sleep 3 s) and a **fast** task (e.g. sleep 0.2 s). Get the fast result, then call **future.cancel()** on the slow task’s future. Print the return value of `cancel()` (True if cancelled, False if already running) and optionally try to get the slow result (it may raise CancelledError).

Requirements:

- Two futures: one long, one short.
- After the short one completes, call `.cancel()` on the long one and print the result.

Hint: Cancel only works if the task hasn’t started running yet; with 2 workers the slow task may already be running, so cancel() often returns False.

In [85]:
# Exercise 9 – cancel()

from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import wait, FIRST_COMPLETED

delays = [5, 3, 4, 7]
def sleep(n: float):
    time.sleep(n)
    return int(time.time() * 1e6)

### Solution 1: without "wait for first completed"
Will label unfinished processes as cancelled, but won't literally cancel their runtimes.

In [86]:
future: Future = None
futures = list()
t0 = int(time.time() * 1e6)

done, cancelled = list(), list()
with ThreadPoolExecutor(4) as pool:
    for n, delay in enumerate(delays):
        futures.append(pool.submit(sleep, delay))
    pending = True
    while pending:
        time.sleep(0.01)
        for n, future in enumerate(futures):
            pending = pending and not future.done()

    for future in futures: future.cancel()
    for n, future in enumerate(futures):
        if future.done(): done.append(n)
        else: cancelled.append(n)
        
delay = int(time.time() * 1e6) - t0

print(f"Done: {done}... Cancelled: {cancelled}... Delay: {delay:.0f} us")

Done: [1]... Cancelled: [0, 2, 3]... Delay: 7001613 us


### Solution 2: with "wait for first completed"

Will cancel the process runtimes and literally stop them, but "`future.cancel()`" won't return anything. So "`cancelled`" array needs to be manually filled with indices not belonging to the "`done`" array.


In [95]:
future: Future = None
futures = list()
t0 = int(time.time() * 1e6)

pool = ThreadPoolExecutor(4)
try:
    futures = [pool.submit(sleep, delay) for delay in delays]
    done, pending = wait(futures, return_when = FIRST_COMPLETED)
    key_done, keys_cancelled = None, list()
    for n, future in enumerate(futures):
        if (future in done): key_done = n
        elif (future in pending):
            keys_cancelled.append(n)
            future.cancel()
finally:
    pool.shutdown(wait = False)

delay_us = int(time.time() * 1e6) - t0

print(f"Done: {key_done}... Cancelled: {keys_cancelled}... Delay: {delay_us:.0f} us")

# Demo: slow + fast; print fast result, then cancel(slow) and its return value

Done: 1... Cancelled: [0, 2, 3]... Delay: 3001273 us


## Exercise 10 – I/O and Timeout Together

**Goal**: Combine **ThreadPoolExecutor** with **timeout**: submit several I/O tasks (different delays). Use **as_completed(futures, timeout=T)** with a timeout so that if not all finish within T seconds, you stop iterating. For each completed future, print the result; if you get **TimeoutError** from `as_completed`, print how many completed and how many are still pending.

Requirements:

- Multiple tasks, e.g. delays [0.5, 0.5, 0.5, 0.5] and timeout=1.0 so only some complete.
- Loop over `as_completed(futures, timeout=...)` and catch TimeoutError to break and report.

Hint: When as_completed times out, it raises TimeoutError; the futures that didn’t complete are still pending (you can count them or check future.done()).

In [111]:
# Exercise 10 – as_completed with timeout

from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed, Future

delays = [0.5, 0.5, 4, 0.5]
timeout = 2 * min(delays)

def sleep(n: float):
    time.sleep(n := abs(n))
    return int(time.time() * 1e6)

futures = dict()
done = dict()
failed = list()
future: Future = None
t0 = int(time.time() * 1e6)
pool = ThreadPoolExecutor(4)
try:
    for n, delay in enumerate(delays):
        futures[pool.submit(sleep, delay)] = n
    for future in as_completed(futures, timeout):
        done[futures[future]] = future.result() - t0
except Exception as EXC: print(repr(EXC))
finally:
    pool.shutdown(wait = False)
    for future in futures:
        if future.done(): continue
        failed.append(futures[future])

delay = int(time.time() * 1e6) - t0

print(f"Done: {done}... Failed: {failed}")
print(f"Timeout: {timeout * 1e6:.0f} us... Delay: {delay} us")

# Demo: 4 tasks × 0.5 s, timeout 1.0 s; show completed count and pending

TimeoutError('1 (of 4) futures unfinished')
Done: {0: 500475, 1: 501038, 3: 501075}... Failed: [2]
Timeout: 1000000 us... Delay: 1006655 us
